# SCIDOCS + HyPE / QuOTE (Hypothetical Prompt Embeddings)

**Context-Enriched Retrieval for RAG — Iris.ai**

HyPE enriches the **chunk side** instead of the query side. At **index time**, an LLM generates
several hypothetical *questions that each chunk answers*; those questions are embedded and indexed,
each pointing back to its source chunk. At **query time** the real user question is matched against
these hypothetical questions — **question ↔ question** matching (symmetric, same distribution).

**Why this is attractive vs. HyDE**
- The LLM cost is paid **once, offline** (indexing). Query time is just a normal embed — **no per-query LLM call**, so it's fast and cheap to serve.
- Matching short query to short question avoids the query↔document asymmetry HyDE fights.

**Two models (same split of roles as HyDE):** generator = OpenAI `gpt-4o-mini` (writes the questions),
embedder = `Octen/Octen-Embedding-0.6B` (fixed). Ends with a baseline-vs-HyPE comparison
(deltas, win/loss, Wilcoxon).

> Note: this generates questions for **all 3,633 corpus docs** (N each), so the generation step is
> heavier than HyDE's (which only touched 323 queries). It is cached + resumable.

> ⚠️ **Heavy on SCIDOCS:** 25,657 docs × 3 questions ≈ 77k LLM calls — expect a long run (hours) and a few dollars. Cached/resumable: if a cell errors, just re-run it.

## 1. Config & setup

In [1]:
import os, json, time, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import pytrec_eval
from scipy import stats
from dotenv import load_dotenv
from openai import OpenAI

MODEL_NAME    = "Octen/Octen-Embedding-0.6B"   # embedder (fixed)
GEN_MODEL     = "gpt-4o-mini"                   # generator (index-time only)
DATA_DIR      = Path("../scidocs")
SPLIT         = "test"
BASELINE_DIR  = Path("results/scidocs/baseline")
OUT_DIR       = Path("results/scidocs/hype")
TOP_K         = 1000
BATCH_SIZE    = 32
SEED          = 42
RUN_TAG       = "hype"

# ---- HyPE knobs ----------------------------------------------------------
N_QUESTIONS   = 3            # hypothetical questions generated per chunk
AGG           = "max"        # how to pool a doc's question scores: 'max' or 'mean'
INCLUDE_DOC   = False        # also index the original chunk embedding as an extra key
GEN_WORKERS   = 4           # parallel OpenAI calls
DOC_CHARS     = 1200         # truncate chunk text fed to the generator

OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
load_dotenv(); client = OpenAI(max_retries=8)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — create a .env file with it."
print("device   :", DEVICE, "| generator:", GEN_MODEL, "| embedder:", MODEL_NAME)

/Users/katerina_dimitrova/Downloads/Iris.ai RAG research/HyDE context Enrichment test/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device   : mps | generator: gpt-4o-mini | embedder: Octen/Octen-Embedding-0.6B


## 2. Load corpus, queries, qrels

In [2]:
def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def load_qrels(p):
    q = {}
    with open(p, encoding="utf-8") as f:
        next(f)
        for line in f:
            qid, did, s = line.rstrip("\n").split("\t")
            if int(s) > 0:
                q.setdefault(qid, {})[did] = int(s)
    return q

corpus = {r["_id"]: ((r.get("title") or "") + "\n" + (r.get("text") or "")).strip()
          for r in load_jsonl(DATA_DIR / "corpus.jsonl")}
corpus_ids = list(corpus.keys())
qrels = load_qrels(DATA_DIR / "qrels" / f"{SPLIT}.tsv")
all_q = {r["_id"]: r["text"] for r in load_jsonl(DATA_DIR / "queries.jsonl")}
queries = {q: all_q[q] for q in qrels if q in all_q}
query_ids = list(queries.keys())
print(f"corpus {len(corpus)} docs | queries {len(queries)}")

corpus 3633 docs | queries 323


## 3. Generate hypothetical questions per chunk (index-time, cached + parallel)

Cached to `results/scidocs/hype/hype_questions.json` keyed by doc id. Resumable — re-runs only fill gaps.
This is the heavy step (~`N_QUESTIONS` x 3,633 calls).

In [3]:
QGEN_PROMPT = (
    "You are given a passage from a scientific paper corpus. Generate exactly {n} diverse, "
    "self-contained questions that this passage directly answers. Vary the phrasing and use "
    "natural wording a researcher might search with. Output ONLY the questions, one per line, no "
    "numbering.\n\nPassage:\n{passage}\n\nQuestions:"
)

def gen_questions(passage, n, retries=4):
    msg = [{"role": "user", "content": QGEN_PROMPT.format(n=n, passage=passage[:DOC_CHARS])}]
    for attempt in range(retries):
        try:
            r = client.chat.completions.create(model=GEN_MODEL, messages=msg,
                                               temperature=0.7, max_tokens=256)
            lines = [ln.strip(" -*0123456789.)\t") for ln in r.choices[0].message.content.splitlines()]
            qs = [ln for ln in lines if len(ln) > 5]
            return qs[:n]
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 * (attempt + 1))

QPATH = OUT_DIR / "hype_questions.json"
questions = json.loads(QPATH.read_text()) if QPATH.exists() else {}
todo = [d for d in corpus_ids if len(questions.get(d, [])) < N_QUESTIONS]
print(f"to generate: {len(todo)} docs  (cached: {len(corpus_ids) - len(todo)})")

lock = threading.Lock(); done = 0
with ThreadPoolExecutor(max_workers=GEN_WORKERS) as ex:
    futs = {ex.submit(gen_questions, corpus[d], N_QUESTIONS): d for d in todo}
    for fut in tqdm(as_completed(futs), total=len(futs)):
        d = futs[fut]
        try:
            qs = fut.result()
        except Exception:
            continue
        with lock:
            questions[d] = qs; done += 1
            if done % 200 == 0:
                QPATH.write_text(json.dumps(questions))
QPATH.write_text(json.dumps(questions))
print("done. example for", corpus_ids[0], "->", questions[corpus_ids[0]][:3])

to generate: 3633 docs  (cached: 0)


100%|██████████| 3633/3633 [08:33<00:00,  7.07it/s]

done. example for MED-10 -> ['Do statins help improve survival rates for breast cancer patients?', 'What was the main finding of the study on statin use and breast cancer mortality?', 'How many breast cancer patients in Finland were included in the study?']


## 4. Embed all hypothetical questions (query space)

Both the indexed questions and the real queries are embedded with the model's **query** prompt, so the
match is question↔question in one consistent space.

In [4]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 256

def embed(texts, prompt_name):
    return model.encode(texts, prompt_name=prompt_name, batch_size=BATCH_SIZE,
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=True).astype(np.float32)

# flat list of all hypothetical questions + which doc each belongs to
q_texts, q_owner = [], []
for d in corpus_ids:
    for q in questions[d][:N_QUESTIONS]:
        q_texts.append(q); q_owner.append(d)
doc_index = {d: i for i, d in enumerate(corpus_ids)}
owner_idx = np.array([doc_index[d] for d in q_owner])   # (M,) doc index per question

print(f"embedding {len(q_texts)} hypothetical questions...")
key_emb = embed(q_texts, "query")          # (M, 1024) the searchable keys

if INCLUDE_DOC:
    corpus_emb = np.load(BASELINE_DIR / "corpus_emb.npy")   # original chunk vectors
    key_emb = np.vstack([key_emb, corpus_emb])
    owner_idx = np.concatenate([owner_idx, np.arange(len(corpus_ids))])
print("index keys:", key_emb.shape[0], "| docs:", len(corpus_ids))

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5107.77it/s]


embedding 18165 hypothetical questions...


Batches: 100%|██████████| 568/568 [03:02<00:00,  3.11it/s]

index keys: 18165 | docs: 3633


## 5. Search: query → keys → pool to docs

Cosine of each query against every hypothetical-question key, then pool the keys back to their doc
(`max` = a doc wins if *any* of its questions matches well).

In [5]:
q_emb = embed([queries[q] for q in query_ids], "query")          # (Q, 1024)
sims = q_emb @ key_emb.T                                          # (Q, M)
Q, D = len(query_ids), len(corpus_ids)

# pool key scores -> per-doc score
if AGG == "max":
    doc_scores = np.full((Q, D), -np.inf, dtype=np.float32)
    rows = np.broadcast_to(np.arange(Q)[:, None], sims.shape)
    cols = np.broadcast_to(owner_idx[None, :], sims.shape)
    np.maximum.at(doc_scores, (rows, cols), sims)
else:  # mean over a doc's questions
    doc_scores = np.zeros((Q, D), dtype=np.float32)
    counts = np.zeros(D, dtype=np.float32)
    for j, d in enumerate(owner_idx):
        doc_scores[:, d] += sims[:, j]; counts[d] += 1
    doc_scores /= np.maximum(counts, 1)[None, :]

k = min(TOP_K, D)
topk = np.argpartition(-doc_scores, k - 1, axis=1)[:, :k]
run = {}
for i, qid in enumerate(query_ids):
    idx = topk[i]; order = idx[np.argsort(-doc_scores[i, idx])]
    run[qid] = {corpus_ids[j]: float(doc_scores[i, j]) for j in order}
print("retrieved top", k, "docs for", len(run), "queries")

Batches: 100%|██████████| 11/11 [00:02<00:00,  4.21it/s]


retrieved top 1000 docs for 323 queries


## 6. Evaluate + save

In [6]:
measures = {"ndcg_cut.1,3,5,10", "recall.3,5,10,20,100", "map", "P.10", "recip_rank", "success.1,5,10"}
per_query = pytrec_eval.RelevanceEvaluator(qrels, measures).evaluate(run)
avg = lambda m: float(np.mean([per_query[q][m] for q in per_query]))
metrics = {
    "NDCG@1": avg("ndcg_cut_1"), "NDCG@3": avg("ndcg_cut_3"), "NDCG@5": avg("ndcg_cut_5"), "NDCG@10": avg("ndcg_cut_10"),
    "Recall@3": avg("recall_3"), "Recall@5": avg("recall_5"), "Recall@10": avg("recall_10"),
    "Recall@20": avg("recall_20"), "Recall@100": avg("recall_100"),
    "Hit@1": avg("success_1"), "Hit@5": avg("success_5"), "Hit@10": avg("success_10"),
    "MAP": avg("map"), "P@10": avg("P_10"), "MRR": avg("recip_rank"),
}
hype_pq = {q: per_query[q]["ndcg_cut_10"] for q in per_query}

with open(OUT_DIR / "run.tsv", "w") as f:
    for qid in query_ids:
        for rank, (did, sc) in enumerate(run[qid].items(), start=1):
            f.write(f"{qid}\tQ0\t{did}\t{rank}\t{sc:.6f}\t{RUN_TAG}\n")
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
(OUT_DIR / "per_query_ndcg10.json").write_text(json.dumps(hype_pq, indent=2))
(OUT_DIR / "config.json").write_text(json.dumps({
    "run_tag": RUN_TAG, "enrichment": "HyPE", "model_name": MODEL_NAME, "generator": GEN_MODEL,
    "n_questions": N_QUESTIONS, "agg": AGG, "include_doc": INCLUDE_DOC,
    "n_index_keys": int(key_emb.shape[0]), "dataset": "SCIDOCS", "split": SPLIT,
    "n_queries": len(queries), "top_k": k, "seed": SEED, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
}, indent=2))

HEADLINE = {"NDCG@10", "Recall@5", "Recall@10", "Hit@5"}
print(f"{'metric':<12} {'score':>8}")
print("-" * 26)
for m, v in metrics.items():
    print(f"{m:<12} {v:>8.4f}{'  <-- headline' if m in HEADLINE else ''}")

metric          score
--------------------------
NDCG@1         0.4396
NDCG@3         0.3952
NDCG@5         0.3701
NDCG@10        0.3396  <-- headline
Recall@3       0.1070
Recall@5       0.1290  <-- headline
Recall@10      0.1695  <-- headline
Recall@20      0.2107
Recall@100     0.3273
Hit@1          0.4582
Hit@5          0.6749  <-- headline
Hit@10         0.7337
MAP            0.1739
P@10           0.2511
MRR            0.5602


## 7. Baseline vs HyPE — deltas, win/loss, significance

In [7]:
base_metrics = json.loads((BASELINE_DIR / "metrics.json").read_text())
base_pq = json.loads((BASELINE_DIR / "per_query_ndcg10.json").read_text())

print(f"{'metric':<12} {'baseline':>9} {'HyPE':>9} {'delta':>9}")
print("-" * 42)
for m in metrics:
    b, h = base_metrics.get(m, float('nan')), metrics[m]; d = h - b
    print(f"{m:<12} {b:>9.4f} {h:>9.4f} {d:>+9.4f}{'  ^' if d>0 else ('  v' if d<0 else '')}")

qs = [q for q in query_ids if q in base_pq]
bb = np.array([base_pq[q] for q in qs]); hh = np.array([hype_pq[q] for q in qs]); diff = hh - bb
wins, losses = int((diff > 1e-9).sum()), int((diff < -1e-9).sum())
ties = len(qs) - wins - losses
print(f"\nNDCG@10 per-query (n={len(qs)}):  wins={wins}  losses={losses}  ties={ties}  mean delta={diff.mean():+.4f}")
print(f"rescued (0->>0): {int(((bb==0)&(hh>0)).sum())} | killed (>0->0): {int(((bb>0)&(hh==0)).sum())}")
nz = diff[np.abs(diff) > 1e-9]
if len(nz):
    st, p = stats.wilcoxon(nz)
    print(f"Wilcoxon: stat={st:.1f}, p={p:.4g} -> {'SIGNIFICANT' if p<0.05 else 'not significant'} at alpha=0.05")

order = np.argsort(diff)
print("\nBiggest HyPE WINS:")
for i in order[::-1][:5]:
    print(f"  {diff[i]:+.3f}  {queries[qs[i]][:70]}")
print("Biggest HyPE LOSSES:")
for i in order[:5]:
    print(f"  {diff[i]:+.3f}  {queries[qs[i]][:70]}")

metric        baseline      HyPE     delta
------------------------------------------
NDCG@1          0.4737    0.4396   -0.0341  v
NDCG@3          0.4206    0.3952   -0.0254  v
NDCG@5          0.4002    0.3701   -0.0301  v
NDCG@10         0.3654    0.3396   -0.0258  v
Recall@3        0.1112    0.1070   -0.0042  v
Recall@5        0.1402    0.1290   -0.0112  v
Recall@10       0.1747    0.1695   -0.0052  v
Recall@20       0.2167    0.2107   -0.0060  v
Recall@100      0.3424    0.3273   -0.0151  v
Hit@1           0.4923    0.4582   -0.0341  v
Hit@5           0.6842    0.6749   -0.0093  v
Hit@10          0.7245    0.7337   +0.0093  ^
MAP             0.1911    0.1739   -0.0172  v
P@10            0.2687    0.2511   -0.0176  v
MRR             0.5851    0.5602   -0.0249  v

NDCG@10 per-query (n=323):  wins=86  losses=121  ties=116  mean delta=-0.0258
rescued (0->>0): 11 | killed (>0->0): 8
Wilcoxon: stat=7915.0, p=0.0009605 -> SIGNIFICANT at alpha=0.05

Biggest HyPE WINS:
  +0.631  myelopathy
